<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 210%;text-align: center;border-radius: 10px 70px">
    AI Ontario Energy Intelligence Dashboard
    
</center></p></h1>

In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout,GRU,Conv1D, MaxPooling1D, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam


import ollama

<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
      Data Loading  
</center></p></h1>

In [ ]:
file = 'ontario_electricity_demand.csv'

df = pd.read_csv(file)

df.head()


,date1,date,hour,hourly_demand,hourly_average_price
0,2002-05-01,2005-05-01,1,14137000,22.97
1,2002-05-01,2005-05-01,2,13872000,23.27
2,2002-05-01,2005-05-01,3,13820000,24.54
3,2002-05-01,2005-05-01,4,13744000,15.17
4,2002-05-01,2005-05-01,5,14224000,23.59


In [4]:
df.drop(columns=["date1"], inplace=True)

In [5]:
df.head()

,date,hour,hourly_demand,hourly_average_price
0,2005-05-01,1,14137000,22.97
1,2005-05-01,2,13872000,23.27
2,2005-05-01,3,13820000,24.54
3,2005-05-01,4,13744000,15.17
4,2005-05-01,5,14224000,23.59


In [6]:
df.to_csv("ontario_electricity.csv", index=False)

In [ ]:

def check_df(df, name="Dataset", head=5):
    print("\n" + "="*60)
    print(f" DATASET OVERVIEW: {name}")
    print("="*60)

    # 1. Shape
    print("\n SHAPE")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

    # 2. Data Types (per column)
    print("\n DATA TYPES (PER COLUMN)")
    print(df.dtypes)

    # 3. Data Type Summary
    print("\n DATA TYPE SUMMARY")
    print(df.dtypes.value_counts())

    # 4. Info
    print("\n INFO")
    df.info()

    # 5. Head
    print("\n HEAD")
    display(df.head(head))

    # 6. Tail
    print("\n TAIL")
    display(df.tail(head))

    # 7. Missing Values
    print("\n MISSING VALUES")
    print(df.isnull().sum())

    # 8. Duplicate Rows
    print("\n DUPLICATE ROWS")
    print(f"Number of duplicate rows: {df.duplicated().sum()}")

    # 9. Descriptive Statistics (numeric only)
    print("\n DESCRIPTIVE STATISTICS (NUMERIC)")
    display(df.describe())


In [8]:
check_df(df, name="Ontario Electricity Demand Dataset")


📊 DATASET OVERVIEW: Ontario Electricity Demand Dataset

📌 SHAPE
Rows: 183432 | Columns: 4

📌 DATA TYPES (PER COLUMN)
date                     object
hour                      int64
hourly_demand             int64
hourly_average_price    float64
dtype: object

📌 DATA TYPE SUMMARY
int64      2
object     1
float64    1
Name: count, dtype: int64

📌 INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183432 entries, 0 to 183431
Data columns (total 4 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   date                  183432 non-null  object 
 1   hour                  183432 non-null  int64  
 2   hourly_demand         183432 non-null  int64  
 3   hourly_average_price  183432 non-null  float64
dtypes: float64(1), int64(2), object(1)
memory usage: 5.6+ MB

📌 HEAD


,date,hour,hourly_demand,hourly_average_price
0,2005-05-01,1,14137000,22.97
1,2005-05-01,2,13872000,23.27
2,2005-05-01,3,13820000,24.54
3,2005-05-01,4,13744000,15.17
4,2005-05-01,5,14224000,23.59



📌 TAIL


,date,hour,hourly_demand,hourly_average_price
183427,2026-04-03,20,16590000,27.78
183428,2026-04-03,21,16140000,38.68
183429,2026-04-03,22,15434000,82.19
183430,2026-04-03,23,14355000,36.37
183431,2026-04-03,24,13755000,33.56



📌 MISSING VALUES
date                    0
hour                    0
hourly_demand           0
hourly_average_price    0
dtype: int64

📌 DUPLICATE ROWS
Number of duplicate rows: 0

📌 DESCRIPTIVE STATISTICS (NUMERIC)


,hour,hourly_demand,hourly_average_price
count,183432.000000,1.834320e+05,183432.000000
mean,12.500000,1.624913e+07,33.812201
std,6.922205,2.592200e+06,33.905004
min,1.000000,2.270000e+06,-138.790000
25%,6.750000,1.430700e+07,14.380000
50%,12.500000,1.614600e+07,29.755000
75%,18.250000,1.803900e+07,43.090000
max,24.000000,2.700500e+07,1891.140000


<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
      clean_data 
</center></p></h1>

In [ ]:
def clean_data(df):

    df = df.copy()
    
    #  TRACK INITIAL STATE
    initial_rows = len(df)
    
    
    #  REMOVE DUPLICATES
    duplicates = df.duplicated().sum()
    df = df.drop_duplicates()
    
    #  CONVERT NUMERIC COLUMNS
    numeric_cols = ['hour', 'hourly_demand', 'hourly_average_price']
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    

    #  TRACK MISSING VALUES (BEFORE)
    missing_before = df[['hourly_demand', 'hourly_average_price']].isnull().sum()
    
    # FILL MISSING VALUES (MULTI-STRATEGY)
    # Interpolate (linear for time series)
    df['hourly_demand'] = df['hourly_demand'].interpolate(method='linear')
    df['hourly_average_price'] = df['hourly_average_price'].interpolate(method='linear')
    
    # Forward fill for any remaining gaps
    df['hourly_demand'] = df['hourly_demand'].ffill()
    df['hourly_average_price'] = df['hourly_average_price'].ffill()
    
    # Backward fill for any gaps at the start
    df['hourly_demand'] = df['hourly_demand'].bfill()
    df['hourly_average_price'] = df['hourly_average_price'].bfill()
    
    #  REMOVE OUTLIERS (Z-SCORE METHOD)
    # Calculate Z-scores for numeric columns
    numeric_df = df[['hourly_demand', 'hourly_average_price']].copy()
    z_scores = np.abs(stats.zscore(numeric_df, nan_policy='omit'))
    
    # Count outliers before removal
    outliers_before = (z_scores > 3).sum()
    
    # Keep rows where both columns have |z-score| < 3
    mask = (z_scores < 3).all(axis=1)
    outliers_removed = (~mask).sum()
    df = df[mask]
    
    # TRACK MISSING VALUES (AFTER)
    missing_after = df[['hourly_demand', 'hourly_average_price']].isnull().sum()
    
    # 9. FINAL VALIDATION
    # Check for any remaining issues
    has_nan = df[['hourly_demand', 'hourly_average_price']].isnull().any().any()
    
    # 10. CREATE CLEANING REPORT
    report = {
        "status": "success" if not has_nan else "partial",
        "initial_rows": initial_rows,
        "final_rows": len(df),
        "rows_removed": initial_rows - len(df),
        "duplicates_removed": int(duplicates),
        "outliers_removed": int(outliers_removed) if outliers_removed > 0 else 0,
        "missing_before_demand": int(missing_before.get('hourly_demand', 0)),
        "missing_before_price": int(missing_before.get('hourly_average_price', 0)),
        "missing_after_demand": int(missing_after.get('hourly_demand', 0)),
        "missing_after_price": int(missing_after.get('hourly_average_price', 0)),
        "has_nan_remaining": has_nan,
        "cleaning_percentage": round((1 - len(df)/initial_rows) * 100, 2)}
    
    return df, report

In [10]:
df, report = clean_data(df)
print("\nCLEANING REPORT:")
for key, value in report.items():
    print(f"{key}: {value}")


CLEANING REPORT:
status: success
initial_rows: 183432
final_rows: 180843
rows_removed: 2589
duplicates_removed: 0
outliers_removed: 2589
missing_before_demand: 0
missing_before_price: 0
missing_after_demand: 0
missing_after_price: 0
has_nan_remaining: False
cleaning_percentage: 1.41


<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
     statistical_summary
</center></p></h1>

In [ ]:
def statistical_summary(df):
    target = "hourly_average_price"
    
    return pd.DataFrame([{
        # Demand
        "mean_demand": df['hourly_demand'].mean(),
        "max_demand": df['hourly_demand'].max(),
        "min_demand": df['hourly_demand'].min(),
        "std_demand": df['hourly_demand'].std(),
        
        # Price
        "mean_price": df[target].mean(),
        "std_price": df[target].std(),
        "min_price": df[target].min(),
        "max_price": df[target].max(),
        "median_price": df[target].median(),
        "q1_price": df[target].quantile(0.25),
        "q3_price": df[target].quantile(0.75),
        "skewness": df[target].skew(),
        "kurtosis": df[target].kurtosis(),
        "volatility": df[target].std(),
        "cv": df[target].std() / df[target].mean() if df[target].mean() > 0 else 0}])

In [12]:
results = statistical_summary(df)
print("\nSTATISTICAL SUMMARY:")
display(results)


STATISTICAL SUMMARY:


,mean_demand,max_demand,min_demand,std_demand,mean_price,std_price,min_price,max_price,median_price,q1_price,q3_price,skewness,kurtosis,volatility,cv
0,1.619085e+07,24014000,8793000,2.530200e+06,31.795536,24.058949,-67.08,135.46,29.47,14.36,42.27,1.031978,1.557756,24.058949,0.756677


<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
    feature_engineering
</center></p></h1>

In [13]:
df['datetime'] = pd.to_datetime(df['date']) + pd.to_timedelta(df['hour'], unit='h')
df = df.sort_values('datetime').set_index('datetime')

In [14]:
df.head()

,date,hour,hourly_demand,hourly_average_price
datetime,,,,
2005-05-01 01:00:00,2005-05-01,1,14137000,22.97
2005-05-01 02:00:00,2005-05-01,2,13872000,23.27
2005-05-01 03:00:00,2005-05-01,3,13820000,24.54
2005-05-01 04:00:00,2005-05-01,4,13744000,15.17
2005-05-01 05:00:00,2005-05-01,5,14224000,23.59


In [17]:

def feature_engineering(df):
    df = df.copy()

    
    df['hour'] = df.index.hour
    df['weekday'] = df.index.dayofweek
    df['month'] = df.index.month
    df['weekend'] = df['weekday'].isin([5, 6]).astype(int)
    
    df['demand_lag_24'] = df['hourly_demand'].shift(24)
    df['price_lag_24'] = df['hourly_average_price'].shift(24)
    
    df['rolling_demand_24'] = df['hourly_demand'].rolling(24).mean()
    df['rolling_price_24'] = df['hourly_average_price'].rolling(24).mean()
    
    # Volatility feature
    df['volatility_24h'] = df['hourly_average_price'].rolling(24).std() / (df['hourly_average_price'].rolling(24).mean() + 1e-6)
    
    return df.dropna()

In [18]:
con= feature_engineering(df)
con.head()

,date,hour,hourly_demand,hourly_average_price,weekday,month,weekend,demand_lag_24,price_lag_24,rolling_demand_24,rolling_price_24,volatility_24h
datetime,,,,,,,,,,,,
2005-05-02 01:00:00,2005-05-02,1,13941000,18.75,0,5,0,14137000.0,22.97,1.645746e+07,28.212083,0.160428
2005-05-02 02:00:00,2005-05-02,2,13651000,18.89,0,5,0,13872000.0,23.27,1.644825e+07,28.029583,0.171717
2005-05-02 03:00:00,2005-05-02,3,13569000,23.30,0,5,0,13820000.0,24.54,1.643779e+07,27.977917,0.173661
2005-05-02 04:00:00,2005-05-02,4,13610000,20.56,0,5,0,13744000.0,15.17,1.643221e+07,28.202500,0.153800
2005-05-02 05:00:00,2005-05-02,5,14204000,26.35,0,5,0,14224000.0,23.59,1.643138e+07,28.317500,0.149926


In [23]:
def eda_analysis(df):

    con= feature_engineering(df)
    return {
        "hourly_demand": con.groupby(con.index.hour)['hourly_demand'].mean(),
        "hourly_price": con.groupby(con.index.hour)['hourly_average_price'].mean(),
        "weekly_price": con.groupby(con['weekend'])['hourly_average_price'].mean(),
        "monthly_price": con.groupby(con['month'])['hourly_average_price'].mean(),
        "corr": con.corr(numeric_only=True),
        "anomalies": (np.abs(stats.zscore(con['hourly_average_price'])) > 3).sum()
    }

In [27]:
res= eda_analysis(df)


In [29]:
print(res['hourly_price'])  # Access specific result
print(res['anomalies'])

datetime
0     24.404568
1     23.331039
2     21.054045
3     19.954458
4     19.282872
5     19.590868
6     22.464671
7     27.992608
8     33.019744
9     35.317455
10    37.372031
11    38.153086
12    37.946005
13    37.053486
14    35.868186
15    35.055572
16    35.801555
17    38.396244
18    40.379433
19    40.892538
20    40.638358
21    38.797127
22    33.286598
23    28.421506
Name: hourly_average_price, dtype: float64
2461


In [28]:
# Display each component separately
print("Hourly Price:")
print(res['hourly_price'])

print("\nWeekly Price (Weekend vs Weekday):")
print(res['weekly_price'])

print("\nMonthly Price:")
print(res['monthly_price'])

print("\nCorrelation Matrix:")
print(res['corr'])

print(f"\nAnomalies Detected: {res['anomalies']}")

Hourly Price:
datetime
0     24.404568
1     23.331039
2     21.054045
3     19.954458
4     19.282872
5     19.590868
6     22.464671
7     27.992608
8     33.019744
9     35.317455
10    37.372031
11    38.153086
12    37.946005
13    37.053486
14    35.868186
15    35.055572
16    35.801555
17    38.396244
18    40.379433
19    40.892538
20    40.638358
21    38.797127
22    33.286598
23    28.421506
Name: hourly_average_price, dtype: float64

Weekly Price (Weekend vs Weekday):
weekend
0    31.121259
1    33.494876
Name: hourly_average_price, dtype: float64

Monthly Price:
month
1     35.682617
2     36.011316
3     32.493841
4     27.671057
5     25.014639
6     28.480012
7     34.201966
8     36.179707
9     32.284321
10    30.021889
11    30.539681
12    33.092190
Name: hourly_average_price, dtype: float64

Correlation Matrix:
                          hour  hourly_demand  hourly_average_price   weekday  \
hour                  1.000000       0.515270              0.231127 -0.001

<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
   stationarity by Adfuller
</center></p></h1>

In [30]:
def check_stationarity(series):
    result = adfuller(series.dropna())
    return {
        "ADF Statistic": result[0],
        "p-value": result[1],
        "stationary": result[1] < 0.05
    }

In [32]:
stationarity = check_stationarity(df['hourly_average_price'])


print(f"ADF Statistic: {stationarity['ADF Statistic']}") 
print(f"p-value: {stationarity['p-value']}")              
print(f"Is Stationary: {stationarity['stationary']}")    


ADF Statistic: -14.73825480829136
p-value: 2.595003755913389e-27
Is Stationary: True


<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
   FORCASTING - ARIMA
</center></p></h1>

In [33]:
def arima_forecast(df, target='hourly_average_price'):
    series = df[target]
    split = int(len(series) * 0.8)
    train = series.iloc[:split]
    test = series.iloc[split:]
    model = ARIMA(train, order=(2,1,2))
    fit = model.fit()
    forecast = fit.forecast(steps=len(test))
    return {
        "train": train,
        "test": test,
        "forecast": forecast,
        "rmse": np.sqrt(mean_squared_error(test, forecast)),
        "mae": mean_absolute_error(test, forecast),
        "mape": mean_absolute_percentage_error(test, forecast)}

In [34]:
arima = arima_forecast(df)

c:\Users\Amir\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Amir\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Amir\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Amir\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is availab

In [35]:
print(f"RMSE: {arima['rmse']:.4f}")
print(f"MAPE: {arima['mape']:.2%}")

RMSE: 22.3680
MAPE: 868352248611041920.00%


In [37]:
# Calculate your relative error
mean_price = df['hourly_average_price'].mean()
rmse = 22.37
relative_error = (rmse / mean_price) * 100

print(f"Mean Price: {mean_price:.2f}$")
print(f"RMSE: {rmse:.2f}$")
print(f"Relative Error: {relative_error:.1f}%")

Mean Price: 31.80$
RMSE: 22.37$
Relative Error: 70.4%


<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
   FORCASTING - LSTM
   
</center></p></h1>

In [ ]:
def create_sequences(data, seq=48):
    X, y = [], []
    for i in range(len(data)-seq):
        X.append(data[i:i+seq])
        y.append(data[i+seq])
    return np.array(X), np.array(y)

def lstm_forecast(df, col='hourly_average_price', steps=24):

    
    scaler = MinMaxScaler()
    data = scaler.fit_transform(df[[col]])
    X, y = create_sequences(data)
    
    if len(X) < 200:
        return {"note": "Not enough data for LSTM"}
    
    split = int(len(X)*0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    X_train = X_train.reshape(-1, X_train.shape[1], 1)
    X_test = X_test.reshape(-1, X_test.shape[1], 1)
    
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(X_train.shape[1],1)),
        Dropout(0.2),
        LSTM(32),
        Dense(1)])
    
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    pred = model.predict(X_test, verbose=0)
    pred_inv = scaler.inverse_transform(pred)
    y_inv = scaler.inverse_transform(y_test.reshape(-1,1))
    
    return {
        "forecast": pred_inv.flatten(),
        "rmse": np.sqrt(mean_squared_error(y_inv, pred_inv))}


In [45]:
lstm = lstm_forecast(df)
print(f"LSTM RMSE: {lstm['rmse']:.4f}")


KeyboardInterrupt: 

<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
   EDA ANALYTICS WITH PLOTS
</center></p></h1>

In [46]:
def sql_query(df, target_col):
    results = {}
    figures = {}
    target = "hourly_average_price"
    

    # Peak hours
    peak_hours = df.groupby('hour')[target].agg(['mean', 'std']).sort_values('mean', ascending=False)
    results['peak_hours'] = peak_hours.head(5)
    
    # Plot: Peak hours bar chart
    fig1, ax1 = plt.subplots(figsize=(10, 5))
    colors = ['red' if i < 3 else 'steelblue' for i in range(len(peak_hours.head(10)))]
    ax1.bar(peak_hours.head(10).index.astype(str), peak_hours.head(10)['mean'], color=colors)
    ax1.set_xlabel('Hour of Day')
    ax1.set_ylabel('Average Price ($/MWh)')
    ax1.set_title('Top 10 Peak Pricing Hours')
    ax1.grid(True, alpha=0.3)
    figures['peak_hours_plot'] = fig1
    


    # Weekly pattern
    weekly_avg = df.groupby('weekday')[target].agg(['mean', 'std'])
    weekly_avg.index = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    results['weekly_avg'] = weekly_avg
    
    # Plot: Weekly pattern
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.plot(weekly_avg.index, weekly_avg['mean'], marker='o', linewidth=2, markersize=8, color='steelblue')
    ax2.fill_between(range(7), weekly_avg['mean'] - weekly_avg['std'], 
                     weekly_avg['mean'] + weekly_avg['std'], alpha=0.2, color='steelblue')
    ax2.set_xlabel('Day of Week')
    ax2.set_ylabel('Average Price ($/MWh)')
    ax2.set_title('Weekly Price Pattern (with ±1 Std Dev)')
    ax2.grid(True, alpha=0.3)
    figures['weekly_plot'] = fig2
    

    # Monthly pattern
    monthly_stats = df.groupby('month')[target].agg(['mean', 'min', 'max', 'std'])
    results['monthly_stats'] = monthly_stats
    
    # Plot: Monthly pattern
    fig3, ax3 = plt.subplots(figsize=(12, 5))
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    ax3.plot(months[:len(monthly_stats)], monthly_stats['mean'], marker='s', linewidth=2, 
             markersize=6, color='green', label='Average Price')
    ax3.fill_between(range(len(monthly_stats)), monthly_stats['min'], monthly_stats['max'], 
                     alpha=0.2, color='green', label='Price Range')
    ax3.set_xlabel('Month')
    ax3.set_ylabel('Price ($/MWh)')
    ax3.set_title('Monthly Price Pattern')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    figures['monthly_plot'] = fig3
    



    # Hourly price profile
    hourly_profile = df.groupby('hour')[target].agg(['mean', 'std'])
    fig4, ax4 = plt.subplots(figsize=(14, 5))
    ax4.plot(hourly_profile.index, hourly_profile['mean'], marker='o', linewidth=2, color='darkblue')
    ax4.fill_between(hourly_profile.index, 
                     hourly_profile['mean'] - hourly_profile['std'], 
                     hourly_profile['mean'] + hourly_profile['std'], 
                     alpha=0.2, color='darkblue')
    ax4.set_xlabel('Hour of Day')
    ax4.set_ylabel('Price ($/MWh)')
    ax4.set_title('24-Hour Price Profile (with Confidence Band)')
    ax4.axhline(df[target].mean(), color='red', linestyle='--', alpha=0.5, 
                label=f'Daily Avg: ${df[target].mean():.2f}')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    figures['hourly_profile'] = fig4
    



    # Volatility by hour
    if 'volatility_24h' in df.columns:
        hourly_volatility = df.groupby('hour')['volatility_24h'].mean().sort_values(ascending=False)
        results['hourly_volatility'] = hourly_volatility.head(5)
        
        # Plot: Volatility heatmap
        fig5, ax5 = plt.subplots(figsize=(12, 6))
        volatility_pivot = df.groupby([df.index.hour, df.index.dayofweek])['volatility_24h'].mean().unstack()
        sns.heatmap(volatility_pivot, ax=ax5, cmap='RdYlGn_r', annot=True, fmt='.3f')
        ax5.set_xlabel('Day of Week (0=Monday, 6=Sunday)')
        ax5.set_ylabel('Hour of Day')
        ax5.set_title('Volatility Heatmap (Hour × Day of Week)')
        figures['volatility_heatmap'] = fig5
    


    # Weekend vs Weekday
    if 'weekend' in df.columns:
        weekend_avg = df.groupby('weekend')[target].agg(['mean', 'min', 'max'])
        weekend_avg.index = ['Weekday', 'Weekend']
        results['weekend_comparison'] = weekend_avg
        
        fig6, ax6 = plt.subplots(figsize=(8, 6))
        weekend_data = [df[df['weekend']==0][target], df[df['weekend']==1][target]]
        bp = ax6.boxplot(weekend_data, labels=['Weekday', 'Weekend'], patch_artist=True)
        bp['boxes'][0].set_facecolor('lightblue')
        bp['boxes'][1].set_facecolor('lightgreen')
        ax6.set_ylabel('Price ($/MWh)')
        ax6.set_title('Price Distribution: Weekday vs Weekend')
        ax6.grid(True, alpha=0.3)
        figures['weekend_boxplot'] = fig6
    


    # Price quantiles
    results['price_quantiles'] = df[target].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    
    # Distribution plot
    fig7, ax7 = plt.subplots(figsize=(10, 5))
    sns.histplot(df[target], bins=50, kde=True, ax=ax7, color='steelblue')
    ax7.axvline(df[target].mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: ${df[target].mean():.2f}')
    ax7.axvline(df[target].median(), color='orange', linestyle='--', linewidth=2, 
                label=f'Median: ${df[target].median():.2f}')
    ax7.set_xlabel('Price ($/MWh)')
    ax7.set_ylabel('Frequency')
    ax7.set_title('Price Distribution Histogram')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    figures['distribution_plot'] = fig7
    
    return results, figures

In [49]:
 df = feature_engineering(df)

In [51]:
sql_results, sql_figures = sql_query(df, "hourly_average_price")
print("\nSQL Query Results:")
for key, value in sql_results.items():
    print(f"\n{key.upper()}:")
    print(value)



SQL Query Results:

PEAK_HOURS:
           mean        std
hour                      
19    40.893823  25.953325
20    40.639664  25.729165
18    40.380566  25.957456
21    38.798052  24.691676
17    38.396894  24.578212

WEEKLY_AVG:
                mean        std
Monday     33.039121  24.281226
Tuesday    32.181964  24.062450
Wednesday  28.678707  22.538343
Thursday   28.496329  23.120992
Friday     33.250382  24.959007
Saturday   33.706319  24.652116
Sunday     33.283713  24.122607

MONTHLY_STATS:
            mean    min     max        std
month                                     
1      35.682617 -61.75  135.46  23.724434
2      36.011316 -35.40  135.03  23.879069
3      32.493841 -61.42  135.20  23.829447
4      27.671057 -63.28  135.42  24.087956
5      25.006773 -61.29  133.21  22.148266
6      28.480012 -63.37  135.19  23.973365
7      34.201966 -67.08  135.38  24.164961
8      36.179707 -64.94  135.07  25.048632
9      32.284321 -42.39  135.43  24.269327
10     30.021889 -54

In [58]:
sql_results, sql_figures = sql_query(df, "hourly_average_price")

#%%
# Display each plot
for key, fig in sql_figures.items():
    print(f"\n📊 {key}")
    fig.show()  # This will open plot in a window
    input("Press Enter to continue...")  # Wait before showing next
    plt.close(fig)  # Close the figure after displaying


📊 peak_hours_plot

📊 weekly_plot

📊 monthly_plot

📊 hourly_profile

📊 volatility_heatmap

📊 weekend_boxplot

📊 distribution_plot


<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 80%;text-align: center;border-radius: 10px 70px">
   AI CONSULTANT
</center></p></h1>

In [ ]:
def ai_consultant(df, sql_result, arima_result=None, lstm_result=None):
    target = "hourly_average_price"
    
    mean_price = df[target].mean()
    cv = df[target].std() / mean_price
    peak = sql_result["peak_hours"].index.tolist()
    correlation = df['hourly_demand'].corr(df[target])
    
    insights = []
    risks = []
    recs = []
    
    insights.append(f"Peak pricing hours: {peak[:3]}")
    
    if cv > 0.3:
        risks.append(f"⚠️ HIGH volatility (CV={cv:.2f})")
        recs.append("Implement real-time price alerts")
    elif cv > 0.15:
        insights.append(f"Moderate volatility (CV={cv:.2f})")
    else:
        insights.append(f"Stable pricing (CV={cv:.2f})")
    
    if "weekend" in df.columns:
        w = df[df["weekend"]==1][target].mean()
        wd = df[df["weekend"]==0][target].mean()
        if wd > 0:
            discount = ((wd - w) / wd) * 100
            if discount > 10:
                insights.append(f"Weekend discount: {discount:.1f}%")
                recs.append(f"Move operations to weekends for {discount:.0f}% savings")
    
    if arima_result and lstm_result:
        arima_rmse = arima_result.get("rmse")
        lstm_rmse = lstm_result.get("rmse")
        if arima_rmse and lstm_rmse and lstm_rmse < arima_rmse:
            insights.append(f"LSTM outperforms ARIMA")
            recs.append("Deploy LSTM for day-ahead forecasting")
    
    if abs(correlation) > 0.5:
        insights.append(f"Strong demand-price correlation (r={correlation:.2f})")
        recs.append("Demand response programs will yield savings")
    
    summary = {
        "avg_price": mean_price,
        "cv": cv,
        "peak_hours": peak[:3],
        "insights": insights,
        "risks": risks,
        "recommendations": recs,
        "correlation": correlation,
        "data_points": len(df),
        "start_date": df.index.min().date(),
        "end_date": df.index.max().date(),
        "anomalies": (np.abs(stats.zscore(df[target])) > 3).sum()}
    
    prompt = f"""You are a SENIOR ENERGY STRATEGY CONSULTANT.

DATA:
- Average Price: ${summary['avg_price']:.2f}/MWh
- Volatility: {summary['cv']:.2%}
- Peak Hours: {summary['peak_hours']}
- Correlation: {summary['correlation']:.2f}

Write executive report with:
1. EXECUTIVE SUMMARY
2. MARKET OVERVIEW
3. KEY INSIGHTS
4. RISK ASSESSMENT
5. RECOMMENDATIONS
6. CONCLUSION"""
    try:
        res = ollama.chat(model="llama3.2:latest",
                          messages=[{"role": "user", "content": prompt}])
        
        return {"type": " LLM Consultant", "report": res["message"]["content"], "summary": summary}
    
    except Exception as e:
        return {"type": " Rule-Based", "report": None, "summary": summary, "error": str(e)}


In [63]:
# If you just want to test without ARIMA/LSTM
ai_result = ai_consultant(df, sql_results)

# Access the results
print(f"Type: {ai_result['type']}")
print(f"Summary: {ai_result['summary']}")

if ai_result['report']:
    print(f"\n📊 AI REPORT:\n{ai_result['report']}")
else:
    print(f"\n⚠️ Error: {ai_result.get('error', 'Unknown error')}")

Type:  LLM Consultant
Summary: {'avg_price': 31.796215050194974, 'cv': 0.7567559485150301, 'peak_hours': [19, 20, 18], 'insights': ['Peak pricing hours: [19, 20, 18]', 'Strong demand-price correlation (r=0.64)'], 'risks': ['⚠️ HIGH volatility (CV=0.76)'], 'recommendations': ['Implement real-time price alerts', 'Demand response programs will yield savings'], 'correlation': 0.6424269418344714, 'data_points': 180795, 'start_date': datetime.date(2005, 5, 3), 'end_date': datetime.date(2026, 4, 4), 'anomalies': 2461}

📊 AI REPORT:
**EXECUTIVE REPORT**

**EXECUTIVE SUMMARY**

As a Senior Energy Strategy Consultant, I have conducted an analysis of the energy market based on the provided data. The report aims to provide insights into the current market conditions, identify key opportunities and risks, and suggest recommendations for optimizing energy strategy.

**MARKET OVERVIEW**

The average price of electricity is $31.80/MWh, with a volatility rate of 75.68%. This indicates that the market i